# ETL del archivo en crudo `cast.parquet`

## Librerías

In [1]:
import os
import ast
import gc

import pandas as pd

## Extracción

In [ ]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/cast.parquet?raw=true"

cast = pd.read_parquet(
    url, 
    engine='fastparquet')

In [3]:
cast.head()

,cast,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...",11862


Valor en la columna 'cast' de la primera fila. Es una cadena con la forma de una lista de diccionarios.

In [4]:
cast['cast'].iloc[0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

La columna 'id' esta completa.

In [ ]:
cast['id'].isnull().sum()

cast    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay valores duplicados en la columna 'id'.

In [ ]:
cast['id'].duplicated(
    keep='first').sum()

44

Se eliminan los duplicados.

In [7]:
cast.drop_duplicates(
    subset='id', 
    inplace=True
    )

Hay valores unicos.

In [ ]:
cast['id'].duplicated(
    keep='first').sum()

0

### Renombrar la columna 'id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con el dataset 'movies.parquet'.

In [ ]:
cast.rename(
    columns={'id': 'movie_id'}, 
    inplace=True)

Se cambio el nombre.

In [10]:
cast.columns

Index(['cast', 'movie_id'], dtype='object')

### Eliminar listas vacias en la columna 'cast'

Se van a eliminar listas vacías de la columna 'cast' para achicar el tamaño del dataset.

Hay listas vacías.

In [11]:
cast[cast['cast'] == "[]"]

,cast,movie_id
137,[],124639
240,[],43475
393,[],42981
438,[],24257
595,[],124472
...,...,...
45447,[],455661
45452,[],44330
45458,[],122036
45462,[],276895


In [12]:
len(cast[cast['cast'] == "[]"])

2414

Se eliminan las listas vacías.

In [13]:
cast = cast.query('cast != "[]"')

Las listas vacías estan eliminadas.

In [14]:
len(cast[cast['cast'] == "[]"])

0

### Desanidar la columna 'cast'

Se convierte a las cadenas en listas de diccionarios.

In [15]:
cast['cast'] = cast['cast'].apply(
    ast.literal_eval)

Se separan los elementos de las listas en filas.

In [16]:
cast_en_filas = cast.explode('cast')

Se convierten las llaves en columnas.

In [17]:
cast_en_columnas = pd.json_normalize(
    cast_en_filas['cast'])

### Crear un nuevo dataframe 'cast'

Se crea un dataframe con la columna 'movie_id' junto con las nuevas columnas.

In [18]:
cast = cast_en_filas.drop(
    columns='cast').join(
        cast_en_columnas)

Se eliminan los siguientes objetos para liberar memoria.

In [19]:
del cast_en_filas
del cast_en_columnas
gc.collect()

660

## Exploración

Se explora el dataframe.

In [20]:
cast

,movie_id,cast_id,character,credit_id,gender,id,name,order,profile_path
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
0,862,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
...,...,...,...,...,...,...,...,...,...
45474,227506,17,Monica,52fe4534c3a36847f80c2103,1,4174,Claire Forlani,2,/6XIXq8n2epBQBvnbU1BXyNJyPYA.jpg
45474,227506,17,Monica,52fe4534c3a36847f80c2103,1,4174,Claire Forlani,2,/6XIXq8n2epBQBvnbU1BXyNJyPYA.jpg
45474,227506,17,Monica,52fe4534c3a36847f80c2103,1,4174,Claire Forlani,2,/6XIXq8n2epBQBvnbU1BXyNJyPYA.jpg
45474,227506,17,Monica,52fe4534c3a36847f80c2103,1,4174,Claire Forlani,2,/6XIXq8n2epBQBvnbU1BXyNJyPYA.jpg


Primera fila.

In [21]:
cast.iloc[0]

movie_id                                     862
cast_id                                       14
character                          Woody (voice)
credit_id               52fe4284c3a36847f8024f95
gender                                         2
id                                            31
name                                   Tom Hanks
order                                          0
profile_path    /pQFoyx7rp09CJTAb932F2g8Nlho.jpg
Name: 0, dtype: object

La columna 'name' esta completa.

In [23]:
cast['name'].isnull().sum()

0

## Transformación de los datos desanidados

### Eliminar las columnas innecesarias

Se las elimina porque son inutiles para la funcion get_actor.

Columnas innecesarias.

In [24]:
innecesarias = [
    'cast_id', 
    'credit_id', 
    'gender', 
    'id', 
    'order', 
    'profile_path'
]

Las columnas innecesarias son eliminadas.

In [25]:
cast.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [26]:
set(cast.columns).isdisjoint(set(innecesarias))

True

Se eliminan las columnas inncesarias de la memoria.

In [ ]:
del cast['cast_id']
del cast['credit_id']
del cast['gender']
del cast['id']
del cast['order']
del cast['profile_path']
gc.collect()

Se ven las columnas que quedan.

In [27]:
for columna in cast.columns:
    print(columna)

movie_id
character
name


### Cambiar el tipo de la columna 'movie_id'

La columna 'movie_id' tiene etiquetas. Entonces se lo cambia al tipo object. El resto de los datos tienen el tipo correcto que es object.

Tipos de las columnas:

In [28]:
cast.dtypes

movie_id      int64
character    object
name         object
dtype: object

* Columna 'movie_id':

In [29]:
cast['movie_id'].dtype

dtype('int64')

In [30]:
cast['movie_id'] = cast['movie_id'].astype(str)

In [31]:
cast['movie_id'].dtype

dtype('O')

Tipos de las columnas con la modificación:

In [32]:
cast.dtypes

movie_id     object
character    object
name         object
dtype: object

### Resetear el índice

El índice actual:

In [33]:
cast.index

Int64Index([    0,     0,     0,     0,     0,     0,     0,     0,     0,
                0,
            ...
            45473, 45473, 45473, 45473, 45473, 45474, 45474, 45474, 45474,
            45474],
           dtype='int64', length=562044)

Se resetea el índice:

In [34]:
cast.reset_index(
    drop=True, 
    inplace=True
    )

El índice actualizado:

In [35]:
cast.index

RangeIndex(start=0, stop=562044, step=1)

### Última revisión

In [36]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   movie_id   562044 non-null  object
 1   character  562044 non-null  object
 2   name       562044 non-null  object
dtypes: object(3)
memory usage: 12.9+ MB


## Carga

In [37]:
ruta_actual = os.getcwd()

ruta_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [38]:
ruta_del_proyecto = os.path.dirname(
    os.path.dirname(
        ruta_actual))

ruta_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [39]:
ruta_a_exportar = os.path.join(
    ruta_del_proyecto, 
    'data', 
    'ETL', 
    'cast.parquet')

ruta_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\cast.parquet'

In [40]:
cast.to_parquet(ruta_a_exportar)

Se elimina el dataframe para liberar memoria.

In [41]:
del cast
gc.collect()

45